# Low-slope sections: reference profiles and RiverSP cycles

**Investigation recorded 2 October 2026.** Following the low-slope and floor-sensitivity audits, inspect all seven parent cases with links below $10^{-5}$ m/m: **11, 13, 15, 21, 22, 41, 46**. In particular, compare case 11 over **0–44 km**, not only its short stability-controlling link.

This notebook reads the **promoted, saved fits actually used for the hydraulic contexts**. It does not rerun profile selection, change slopes, or run RAPID. It uses the saved screened observations, not a fresh processing of raw SWOT. Production data must be mounted at the paths recorded in the hydraulic manifests.

Each case has the requested interactive raw-WSE plot, a second stage-aligned view, profile diagnostics, cycle support and acquisition offsets. Use the **section dropdown** to zoom to any flagged link; click legend entries to show individual acquisitions. Dropdown bounds include raw observations, so a raw plot can look flat at full scale: zoom the y-axis or use the aligned plot. Distance increases **upstream**, away from the outlet. Pass IDs and cycle IDs are shown separately.

The support tables are the dated 2 October audit. Assertions check that their flagged link IDs and promoted profile sources still match the loaded inputs. If slopes are subsequently refitted, regenerate the audit before interpreting these tables.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "RAPID/src").is_dir())
AUDIT = ROOT / "RAPID/assets/low_slope_all_cycles_20261002"
CASES = ["11", "13", "15", "21", "22", "41", "46"]
pio.renderers.default = "plotly_mimetype"
support = pd.read_csv(AUDIT / "section_summary.csv", dtype={"example": str})
support["example"] = support.example.str.zfill(2)
provenance = json.loads((AUDIT / "provenance.json").read_text())

def load_case(example):
    suffix = "_direction130" if example == "15" else ""
    context = ROOT / f"network_variants/output/{example}_indep{suffix}__anchored/states/S000_base/hydraulic_context"
    manifest = json.loads((context / "hydraulic_context_manifest.json").read_text())
    source = Path(manifest["slope_refit_promotion"]["reference_profile_dir"])
    assert source.resolve() == Path(provenance[example]).resolve(), "Source changed: regenerate cycle audit"
    profile = pd.read_parquet(source / "reference_profile.parquet").sort_values("reference_section_dist_out_m")
    observations = pd.read_parquet(source / "reference_profile_observations.parquet")
    links = pd.read_csv(context / "link_hydraulic_context.csv")
    flagged = links[links.slope_used < 1e-5].copy()
    sections = support[(support.example == example) & (support.window == "exact")].copy()
    assert set(flagged.id_link) == set(sections.link), "Flagged links changed: regenerate cycle audit"
    for row in sections.itertuples():
        current = flagged.set_index("id_link").loc[row.link, "slope_used"]
        assert np.isclose(current, row.link_slope, rtol=1e-9, atol=1e-14), "Slopes changed: regenerate audit"
    return dict(example=example, source=source, profile=profile, observations=observations,
                offsets=pd.read_parquet(source / "reference_profile_acquisition_offsets.parquet"),
                summary=json.loads((source / "reference_profile_audit.json").read_text())["summary"],
                flagged=flagged, sections=sections)

cases = {ex: load_case(ex) for ex in CASES}
assert sum(len(v["flagged"]) for v in cases.values()) == 57
display(support[support.window == "exact"].groupby(["example", "classification"]).size().unstack(fill_value=0))

## Why can the fitted slope become this low?

The implemented fit represents each observation as a **single common profile plus one constant vertical offset per acquisition**. It minimizes weighted WSE residuals plus a curvature penalty, under a monotonic minimum-gradient constraint. The static SWORD profile provides initialization and the reference slope used for the floor; it is not independent proof that the fitted gradient is correct. See `RAPID/src/rapid_tools/reference_profile.py`, particularly `_fit` and `fit_reference_profile`.

Four effects need to be distinguished:

1. **Different acquisition shapes.** Constant offsets remove stage shifts but cannot make steep and flat longitudinal profiles identical. A common fit compromises between them. With equal observation weights, cycles with more observations can contribute more to fitting; the audit instead gives eligible cycles equal weight.
2. **Weakly observed sections and constraints.** Where local observations are sparse, smoothing and monotonic constraints can determine the shape. Floor-active intervals indicate a binding numerical lower bound, not a directly measured physical slope. Smoothing can redistribute gradient around a break; it does not always flatten every section.
3. **Numerical convergence.** The saved iteration counts are shown below. Reaching an iteration limit warrants checking convergence, but does not by itself prove a fit failed. The cause of a particular low slope cannot be assigned to regularization or convergence without a controlled refit.
4. **Mapping the profile onto branches.** A shared head drop divided by a longer physical branch length gives a smaller branch slope. Compound-domain solvers can also redistribute internal heads. Their link slopes are not the same quantity as the gradient of the reference corridor.

Consequently, a low slope may reflect observation-supported flatness, unresolved information, fitting behavior, or head allocation. This notebook identifies evidence for these possibilities; it does **not** establish a universal physical lower bound. We should resolve profile support before changing Kb or selecting slopes to make routing stable.

### Reading the support tables

Each acquisition is fitted on the link's **projected reference-section span**, using OLS and robust Theil–Sen slopes; the current profile is evaluated at the same sample positions. Eligibility requires at least four positions covering at least half the window. Multiple passes within a cycle are combined before equal-cycle summaries. At least three cycles are required. “Consistent steeper support” requires at least 75% of cycles to be both positive and steeper than the profile under both estimators. This is a descriptive screen, not a confidence interval or a replacement slope.

The eight supported links occupy six distinct locations; shared boundaries are not independent observations. Expanded windows add 1 km at each end and may cross a break. The P10–P90 plot envelope mixes stages and is **not** a slope confidence band.

In [ ]:
def profile_figure(case, *, aligned=False):
    profile = case["profile"]
    obs = case["observations"]
    field = "stage_aligned_wse_m" if aligned else "wse_m"
    envelope = obs.groupby("reference_section_dist_out_m")[field].quantile([.1, .9]).unstack().sort_index()
    fig = go.Figure()
    for acquisition_id, group in obs.groupby("acquisition_id", sort=False):
        group = group.sort_values("reference_section_dist_out_m")
        first = group.iloc[0]
        label = f"Cycle {int(first['cycle']):03d} · pass {int(first['pass']):03d} · {str(first['time'])[:10]}"
        fig.add_scatter(x=group.reference_section_dist_out_m / 1000, y=group[field],
            mode="lines+markers", name=label, visible="legendonly",
            line=dict(width=1), marker=dict(size=3), opacity=.55,
            meta=str(acquisition_id), hovertemplate="%{meta}<br>%{x:.3f} km<br>%{y:.4f} m<extra></extra>")
    fig.add_scatter(x=envelope.index / 1000, y=envelope[.9], mode="lines",
        line=dict(width=0), showlegend=False, hoverinfo="skip")
    fig.add_scatter(x=envelope.index / 1000, y=envelope[.1], mode="lines", fill="tonexty",
        fillcolor="rgba(120,120,120,.18)", line=dict(width=0), name="screened P10–P90 envelope")
    fig.add_scatter(x=profile.reference_section_dist_out_m / 1000, y=profile.fitted_wse_m,
        mode="lines", name="promoted monotone multi-pass profile", line=dict(color="#d62728", width=3))
    fig.add_scatter(x=profile.reference_section_dist_out_m / 1000, y=profile.static_wse_m,
        mode="lines", name="static SWORD P50 reference", line=dict(color="#9467bd", dash="dash"))
    for row in case["sections"].itertuples():
        fig.add_vrect(x0=row.lo/1000, x1=row.hi/1000, fillcolor="#ffbf00", opacity=.10, line_width=0, layer="below")
    options = [("Full corridor", None)]
    if case["example"] == "11":
        options += [("Downstream 0–44 km", (0, 44000)), ("Downstream 0–40 km (exclude break)", (0, 40000))]
    for row in case["sections"].itertuples():
        options.append((f"Link {int(row.link)}: {row.lo/1000:.1f}–{row.hi/1000:.1f} km", (row.lo-1000, row.hi+1000)))
    buttons=[]
    for label, bounds in options:
        update={"xaxis.autorange": True, "yaxis.autorange": True}
        if bounds is not None:
            lo,hi=bounds
            pp=profile[profile.reference_section_dist_out_m.between(lo,hi)]
            oo=obs[obs.reference_section_dist_out_m.between(lo,hi)]
            yy=np.r_[pp.fitted_wse_m, pp.static_wse_m, oo[field]]
            yy=yy[np.isfinite(yy)]
            update={"xaxis.range": [lo/1000,hi/1000], "xaxis.autorange": False}
            if len(yy):
                pad=max(.05, .05*np.ptp(yy));update.update({"yaxis.range": [float(yy.min()-pad),float(yy.max()+pad)], "yaxis.autorange": False})
        buttons.append(dict(label=label, method="relayout", args=[update]))
    fig.update_layout(title=f"Case {case['example']} — {'stage-aligned' if aligned else 'raw'} WSE and promoted profile",
        xaxis_title="Reference-section distance to outlet [km]", yaxis_title="WSE [m]",
        height=610, template="plotly_white", legend=dict(font=dict(size=10)),
        updatemenus=[dict(buttons=buttons, x=0, y=1.17, xanchor="left")], margin=dict(t=110))
    return fig

def inspect_case(example):
    case=cases[example]; summary=case["summary"]
    print("Promoted source:", case["source"])
    profile_figure(case).show()
    profile_figure(case, aligned=True).show()
    intervals=case["profile"].dropna(subset=["interval_slope_m_per_m"])
    print(f"Global profile slope: {summary['fitted_global_slope_m_per_m']:.6g} m/m")
    print(f"Minimum profile slope: {intervals.interval_slope_m_per_m.min():.6g} m/m")
    print(f"Configured minimum: {summary['minimum_slope_m_per_m']:.6g} m/m")
    print(f"Floor-active intervals: {summary['n_floor_active_intervals']}")
    print("Saved iterations (outer, final inner):", summary.get("outer_iterations"), summary.get("inner_iterations"))
    print("Curvature penalty lambda:", summary.get("lambda_value"), "| observation weighting:", summary.get("weight_mode"))
    display(case["sections"][["link", "ownership", "lo", "hi", "link_slope", "n_cycles",
        "profile_theil_median", "theil_median", "theil_q25", "theil_q75", "classification"]])
    expanded=support[(support.example==example)&(support.window=="expanded_1km_each_end")]
    display(Markdown("**Expanded windows: contextual evidence only**"))
    display(expanded[["link", "n_cycles", "profile_theil_median", "theil_median", "classification"]])
    display(Markdown("**Acquisition stage offsets (m)**"))
    labels=case["observations"][["acquisition_id", "cycle", "pass", "time"]].drop_duplicates("acquisition_id")
    display(case["offsets"].merge(labels,on="acquisition_id").sort_values("stage_offset_m"))

## Case 11

The broad 0–44 km region is the main concern. Cycles 005 and 007 are much steeper, but the median across all 11 cycles is near the common profile's broad robust gradient. Select both cycles in the legend and compare them with later cycles. Repeat for 0–40 km to exclude the break. This supports acquisition-dependent shape differences, not a blanket preference for the steepest cycles. The short floor-bound region around 42 km is a different spatial question. The saved fit reaches 100 outer / 500 final-inner iterations; convergence remains to be checked.

In [ ]:
inspect_case("11")

In [ ]:
display(support[(support.example == "11") & (support.link == -1)][[
    "window", "n_cycles", "profile_theil_median", "theil_median", "theil_q25", "theil_q75", "fraction_both_steeper"]])

## Case 13

Links 59 and 107 have repeated support for steeper projected gradients. Link 174, the very-low-slope outlet controller, lacks sufficient exact-span coverage. Thus the three low slopes cannot all be assigned the same explanation: two expose local fit discrepancies; one remains weakly resolved.

In [ ]:
inspect_case("13")

## Case 15

All four flagged links belong to compound geometry and lack sufficient exact-span cycle support. Some projected spans are extremely short or zero. Profile gradients and domain-allocated physical link slopes must be examined separately. Wider-window observations cannot directly validate each internal branch.

In [ ]:
inspect_case("15")

## Case 21

Links 52 and 53 share one projected window and boundary head difference. Ten eligible cycles support a steeper longitudinal gradient than the profile. This is one location with strong evidence to investigate local fitting, not two independent confirmations. A controlled refit is needed to identify why this gradient was suppressed.

In [ ]:
inspect_case("21")

## Case 22

Most flagged spans have insufficient local acquisition coverage; two exact spans have mixed evidence from three eligible cycles. Expanded windows all fail the stated coverage criteria. These low slopes are not validated by an absence of contradictory fits. Inspect observation gaps and compound-domain boundaries before changing floors.

In [ ]:
inspect_case("22")

## Case 41

All three flagged links show repeated support for steeper projected gradients. Links 186 and 196 share one location; link 95 is another. This is a priority for checking local fit residuals, regularization and convergence. The sub-1e-6 link 95 slope should not be treated as well-supported simply because the overall profile was accepted.

In [ ]:
inspect_case("41")

## Case 46

One exact span (link 46) shows consistent steeper support, with mixed evidence or insufficient coverage elsewhere. Wider windows support additional steeper gradients, but may mix neighboring sections. This case reads the promoted bridged-floor-v4 profile, retaining its existing 370–410 km bridge; that imposed segment must not be mistaken for a fresh observation-based local fit.

In [ ]:
inspect_case("46")

## What to resolve before returning to Kb and routing stability

- First check the six distinct locations with repeated steeper support through convergence checks and controlled local-fit sensitivity, assessed on withheld acquisitions and head consistency.
- For case 11, distinguish a common reference profile from the differing hydraulic/retrieval conditions of individual cycles. A steeper reference requires a stated scientific selection criterion, not selection for numerical convenience.
- For unsupported short or compound spans, review domain-scale observations and boundary head losses. Do not transfer a neighboring-window slope directly onto a branch.
- Only then repeat Kb/celerity/stability diagnostics. No physical slope is selected by this notebook.

Full audit: [2 October cycle-support report](../assets/low_slope_all_cycles_20261002/README.md). The notebook is an investigation of the saved fits, not a completed causal diagnosis of their low slopes.